In [43]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import category_encoders as ce
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import optuna
from collections import Counter

In [44]:
df = pd.read_csv('training.csv')
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


In [45]:
df = df.drop(['PRIMEUNIT', 'AUCGUART', 'RefId'], axis=1)

In [46]:
df_new = df.copy()

dates = np.sort(df_new['PurchDate'].unique())
n = len(dates)

train_dates = dates[n // 3]
validation_dates = dates[2 * n // 3]

train_mask = dates <= train_dates
validation_mask = (dates > train_dates) & (dates <= validation_dates)
test_mask = dates > validation_dates

train_part = dates[train_mask]
validation_part = dates[validation_mask]
test_part = dates[test_mask]

X_train = df_new[df_new['PurchDate'].isin(train_part)].drop(columns=['PurchDate', 'IsBadBuy'])
X_valid = df_new[df_new['PurchDate'].isin(validation_part)].drop(columns=['PurchDate', 'IsBadBuy'])
X_test = df_new[df_new['PurchDate'].isin(test_part)].drop(columns=['PurchDate', 'IsBadBuy'])

y_train = df_new.loc[df_new['PurchDate'].isin(train_part), 'IsBadBuy']
y_valid = df_new.loc[df_new['PurchDate'].isin(validation_part), 'IsBadBuy']
y_test = df_new.loc[df_new['PurchDate'].isin(test_part), 'IsBadBuy']

Обработка данных

In [47]:
categorical_features = X_train.select_dtypes(['string', 'string', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(['number']).columns.tolist()
n_unique = X_train[categorical_features].nunique()
low_card = n_unique[n_unique <= 10].index.tolist()
mid_card = n_unique[(n_unique > 10) & (n_unique <= 50)].index.tolist()
high_card = n_unique[n_unique > 50].index.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

low_card_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

mid_card_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

high_card_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', ce.CatBoostEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('low_card', low_card_pipeline, low_card),
        ('mid_card', mid_card_pipeline, mid_card),
        ('high_card', high_card_pipeline, high_card)
    ]                             
)

In [48]:
preprocessor.fit(X_train, y_train)
X_train_new = preprocessor.transform(X_train)
X_valid_new = preprocessor.transform(X_valid)
X_test_new = preprocessor.transform(X_test)

Реализация дерева

In [49]:
class Node():
    def __init__(self, feature=None, left_subtree=None, right_subtree=None, threshold=None, value=None):
        self.feature = feature
        self.left_subtree = left_subtree
        self.right_subtree = right_subtree
        self.threshold = threshold
        self.value = value
    
    def is_leaf(self):
        return self.value is not None


In [50]:
class MyDecisionTreeClassifier:
    def __init__(self, max_depth):
        self.max_depth = max_depth
        self.tree = None
        self.n_classes = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        self.n_classes = len(np.unique(y))
        self.tree = self._build_tree(X, y, depth=0)
        return self
        

    def _build_tree(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            counts = np.bincount(y, minlength=self.n_classes)
            proba = counts / counts.sum()
            return Node(value=proba)
        
        feature, threshold = self._best_split(X, y)

        if feature is None:
            counts = np.bincount(y, minlength=self.n_classes)
            proba = counts / counts.sum()
            return Node(value=proba)
        
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=feature, threshold=threshold, left_subtree=left_subtree, right_subtree=right_subtree)


    def _best_split(self, X, y):
        n_samples, n_features, = X.shape
        best_gini = float('inf')
        best_feature = None
        best_threshold = None 

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]
                gini_left = self._gini(y_left)
                gini_right = self._gini(y_right)
                weighted_gini = len(y_left) / n_samples * gini_left + len(y_right) / n_samples * gini_right

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _gini(self, y):
        if len(y) == 0:
            return 0
        
        counts = np.bincount(y, minlength=self.n_classes)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def predict_proba(self, X):
        X = np.array(X)
        return np.array([self._predict_row(x, self.tree) for x in X])

    def _predict_row(self, x, node):
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_row(x, node.left_subtree)
        else: 
            return self._predict_row(x, node.right_subtree)

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)
        

In [51]:
class MyDecisionTreeRegressor:
    def __init__(self, max_depth):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        self.tree = self._build_tree(X, y, depth=0)
        return self
        

    def _build_tree(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) <= 1:
            value = np.mean(y)
            return Node(value=value)

        feature, threshold = self._best_split(X, y)

        if feature is None:
            value = np.mean(y)
            return Node(value=value)
        
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(feature=feature, threshold=threshold, left_subtree=left_subtree, right_subtree=right_subtree)


    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_mse = float('inf')
        best_feature = None
        best_threshold = None

        for feature in range(n_features):
            values = np.unique(X[:, feature])

            # берем не все пороги, а только часть
            if len(values) > 20:
                idx = np.linspace(0, len(values) - 1, 20, dtype=int)
                thresholds = values[idx]
            else:
                thresholds = values

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                mse_left = self._mse(y_left)
                mse_right = self._mse(y_right)
                weighted_mse = (len(y_left) / n_samples) * mse_left + (len(y_right) / n_samples) * mse_right

                if weighted_mse < best_mse:
                    best_mse = weighted_mse
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def _mse(self, y):
        if len(y) == 0:
            return 0
    
        return np.mean((y - np.mean(y)) ** 2)

    def _predict_row(self, x, node):
        if node.value is not None:
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_row(x, node.left_subtree)
        else: 
            return self._predict_row(x, node.right_subtree)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict_row(x, self.tree) for x in X])
        
    

In [52]:
tree = MyDecisionTreeClassifier(max_depth=2)
tree.fit(X_train_new, y_train)
y_proba = tree.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.3111329740441664


In [53]:
tree = DecisionTreeClassifier(max_depth=2)
tree.fit(X_train_new, y_train)
y_proba = tree.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.3111329740441664


Реализация модели RandomForest

In [54]:
class MyRandomForestClassifier:
    def __init__(self, n_estimators=50, max_depth=2, max_features='sqrt', random_state=21):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_subsets = []
        self.n_classes = None

    def _get_max_features(self, n_features):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif self.max_features is None:
            return n_features
    
    def _bootstrap_sample(self, X, y, rng):
        n_samples = X.shape[0]
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        self.trees = []
        self.features_subsets = []
        self.n_classes = len(np.unique(y))
        rng = np.random.default_rng(self.random_state)

        n_features = X.shape[1]
        n_features_subsample = self._get_max_features(n_features)

        for _ in range(self.n_estimators):
            X_sample, y_sample = self._bootstrap_sample(X, y, rng)
            feature_indices = rng.choice(n_features, size=n_features_subsample, replace=False)
            X_sample_sub = X_sample[:, feature_indices]

            tree = MyDecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X_sample_sub, y_sample)
            self.trees.append(tree)
            self.features_subsets.append(feature_indices)
        
        return self
    
    def predict_proba(self, X):
        X = np.array(X)
        all_probas = []

        for tree, feature_indices in zip(self.trees, self.features_subsets):
            X_sub = X[:, feature_indices]
            probas = tree.predict_proba(X_sub)
            all_probas.append(probas)
        
        all_probas = np.array(all_probas)
        mean_probas = np.mean(all_probas, axis=0)
        return mean_probas

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)


In [55]:
tree = MyRandomForestClassifier(max_depth=2)
tree.fit(X_train_new, y_train)
y_proba = tree.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.42114726313927386


In [56]:
tree = RandomForestClassifier(max_depth=2)
tree.fit(X_train_new, y_train)
y_proba = tree.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.4372835719032526


Реализация модели Градиентный бустинг

In [57]:
class MyGBDTClassifier:
    def __init__(self, max_depth, n_estimators=100, max_features=None, learning_rate=0.1):
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.learning_rate = learning_rate

        self.trees = []
        self.base_score = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        p0 = np.clip(np.mean(y), 1e-15, 1-1e-15)
        self.base_score = np.log(p0 / (1- p0))

        F = np.full(len(y), self.base_score)
        self.trees = []

        for _ in range(self.n_estimators):
            p = self._sigmoid(F)
            gradient = y - p

            tree = MyDecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, gradient)
            update = tree.predict(X)
            F += self.learning_rate * update
            self.trees.append(tree)
        
        return self
    
    def predict_proba(self, X):
        X = np.array(X)
        F = np.full(X.shape[0], self.base_score, dtype=float)

        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        p1 = self._sigmoid(F)
        p0 = 1 - p1

        return np.column_stack((p0, p1))
    
    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)

In [58]:
tree = MyGBDTClassifier(max_depth=2)
tree.fit(X_train_new, y_train)
y_proba = tree.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.44252014278337914


In [59]:
LightGBM = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, num_leaves=4, random_state=21)
LightGBM.fit(X_train_new, y_train)
y_proba = LightGBM.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

[LightGBM] [Info] Number of positive: 2970, number of negative: 21552
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3639
[LightGBM] [Info] Number of data points in the train set: 24522, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.121116 -> initscore=-1.981907
[LightGBM] [Info] Start training from score -1.981907
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
0.4407557443238048


/home/caesar/ML5_Decision_trees.ID_1254804-1/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [60]:
XGBoost = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, booster='dart', rate_drop=0.1, skip_drop=0.5, sample_type='uniform', normalize_type='tree', random_state=21)
XGBoost.fit(X_train_new, y_train)
y_proba = XGBoost.predict_proba(X_valid_new)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.4446761915348316


In [61]:
X_train

,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,RED,AUTO,1.0,...,7451.0,8552.0,11597.0,12409.0,21973,33619,FL,7100.0,0,1113
1,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,1.0,...,7456.0,9222.0,11374.0,12791.0,19638,33619,FL,7600.0,0,1053
2,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,2.0,...,4035.0,5557.0,7146.0,8702.0,19638,33619,FL,4900.0,0,1389
3,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,1.0,...,1844.0,2646.0,4375.0,5518.0,19638,33619,FL,4100.0,0,630
4,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,2.0,...,3247.0,4384.0,6739.0,7911.0,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,BLACK,AUTO,1.0,...,2190.0,3055.0,4836.0,5937.0,18111,30212,GA,4200.0,0,993
72979,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,SILVER,AUTO,NaN,...,6785.0,8132.0,10151.0,11652.0,18881,30212,GA,6200.0,0,1038
72980,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,SILVER,AUTO,1.0,...,8375.0,9802.0,11831.0,14402.0,18111,30212,GA,8200.0,0,1893
72981,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,WHITE,AUTO,1.0,...,6590.0,7684.0,10099.0,11228.0,18881,30212,GA,7000.0,0,1974


In [62]:
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()
X_test_cb = X_test.copy()

X_train_cb[numeric_features] = numeric_pipeline.fit_transform(X_train[numeric_features])
X_valid_cb[numeric_features] = numeric_pipeline.transform(X_valid[numeric_features])
X_test_cb[numeric_features] = numeric_pipeline.transform(X_test[numeric_features])
for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].fillna('Missing').astype(str)
    X_valid_cb[col] = X_valid_cb[col].fillna('Missing').astype(str)
    X_test_cb[col] = X_test_cb[col].fillna('Missing').astype(str)

In [63]:
cb = CatBoostClassifier(iterations=100, depth=3, learning_rate=0.1, verbose=False)
cb.fit(X_train_cb, y_train, cat_features=categorical_features)
y_proba = cb.predict_proba(X_valid_cb)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(gini)

0.48553925347695337


Подбор лучших параметров для CatBoost

In [64]:
optuna.logging.set_verbosity(optuna.logging.ERROR)

In [65]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 50, 100),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10)
    }

    model = CatBoostClassifier(**params)

    model.fit(X_train_cb, y_train, cat_features=categorical_features, eval_set=(X_valid_cb, y_valid), early_stopping_rounds=30, use_best_model=True, verbose=False)
    y_proba = model.predict_proba(X_valid_cb)
    gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
    return gini

In [66]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 24. Best value: 0.490334: 100%|██████████| 50/50 [01:24<00:00,  1.69s/it]


In [67]:
model = CatBoostClassifier(**study.best_params)
model.fit(X_train_cb, y_train, cat_features=categorical_features, verbose=False)
y_proba = cb.predict_proba(X_train_cb)
gini = 2 * roc_auc_score(y_train, y_proba[:, 1]) - 1
print(f'Train gini: {gini}')
y_proba = cb.predict_proba(X_valid_cb)
gini = 2 * roc_auc_score(y_valid, y_proba[:, 1]) - 1
print(f'Valid gini: {gini}')
y_proba = cb.predict_proba(X_test_cb)
gini = 2 * roc_auc_score(y_test, y_proba[:, 1]) - 1
print(f'Test gini: {gini}')


Train gini: 0.5674756879610259
Valid gini: 0.48553925347695337
Test gini: 0.4999440523472789


Можем заметить, что модель несколько переобучилась, что может быть связано с тем, что мы разделили выборку по датам, так как конъюктура рынка могла измениться.

Бонусная часть 

In [68]:
class MyRandomTreeClassifier:
    def __init__(self, max_depth=3, random_state=21):
        self.max_depth = max_depth
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)
        self.tree = None
        self.n_classes = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        self.n_classes = len(np.unique(y))
        self.tree = self._build_tree(X, y, depth=0)
        return self

    def _build_tree(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            counts = np.bincount(y, minlength=self.n_classes)
            proba = counts / counts.sum()
            return Node(value=proba)

        feature, threshold = self._best_split(X, y)

        if feature is None:
            counts = np.bincount(y, minlength=self.n_classes)
            proba = counts / counts.sum()
            return Node(value=proba)

        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(
            feature=feature,
            threshold=threshold,
            left_subtree=left_subtree,
            right_subtree=right_subtree
        )

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_gini = float('inf')
        best_feature = None
        best_threshold = None

        for feature in range(n_features):
            col = X[:, feature]
            col_min = np.min(col)
            col_max = np.max(col)

            if col_min == col_max:
                continue

            threshold = self.rng.uniform(col_min, col_max)

            left_mask = X[:, feature] <= threshold
            right_mask = X[:, feature] > threshold

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            y_left = y[left_mask]
            y_right = y[right_mask]

            gini_left = self._gini(y_left)
            gini_right = self._gini(y_right)

            weighted_gini = (
                len(y_left) / n_samples * gini_left
                + len(y_right) / n_samples * gini_right
            )

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

        return best_feature, best_threshold

    def _gini(self, y):
        if len(y) == 0:
            return 0

        counts = np.bincount(y, minlength=self.n_classes)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def predict_proba(self, X):
        X = np.array(X)
        return np.array([self._predict_row(x, self.tree) for x in X])

    def _predict_row(self, x, node):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict_row(x, node.left_subtree)
        else:
            return self._predict_row(x, node.right_subtree)

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

In [69]:
class MyExtraTreesClassifier:
    def __init__(self, n_estimators=50, max_depth=2, max_features='sqrt', random_state=21):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_subsets = []
        self.n_classes = None

    def _get_max_features(self, n_features):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif isinstance(self.max_features, float) and self.max_features < 1:
            return max(1, int(n_features * self.max_features))
        elif self.max_features is None:
            return n_features

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        self.trees = []
        self.features_subsets = []
        self.n_classes = len(np.unique(y))
        rng = np.random.default_rng(self.random_state)

        n_features = X.shape[1]
        n_features_subsample = self._get_max_features(n_features)

        for i in range(self.n_estimators):
            feature_indices = rng.choice(n_features, size=n_features_subsample, replace=False)
            X_sub = X[:, feature_indices]

            tree = MyRandomTreeClassifier(
                max_depth=self.max_depth,
                random_state=self.random_state + i
            )
            tree.fit(X_sub, y)

            self.trees.append(tree)
            self.features_subsets.append(feature_indices)

        return self

    def predict_proba(self, X):
        X = np.array(X)
        all_probas = []

        for tree, feature_indices in zip(self.trees, self.features_subsets):
            X_sub = X[:, feature_indices]
            probas = tree.predict_proba(X_sub)
            all_probas.append(probas)

        all_probas = np.array(all_probas)
        mean_probas = np.mean(all_probas, axis=0)
        return mean_probas

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.argmax(probas, axis=1)

In [70]:
model = MyExtraTreesClassifier()
model.fit(X_train_new, y_train)
y_proba = model.predict_proba(X_test_new)
gini = 2 * roc_auc_score(y_test, y_proba[:, 1]) - 1

In [71]:
gini

0.4746421242633565

Плюсы данного алгоритма:  
1.Высокая устойчивость  
2.Быстрое обучение  
Минусы данного алгоритма:  
1.Снижение точности из-за случайности  
2.Используется не оптимальный split  